In [5]:
conn = duckdb.connect(":memory:")

# Configure MinIO connection
conn.execute("""
    INSTALL s3;
    LOAD s3;
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='admin';
    SET s3_secret_access_key='password123';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

# GOLD LOGIC: Aggregate metrics by Vendor
# We calculate total revenue and average fare
gold_sql = """
    COPY (
        SELECT
            vendor_id,
            count(*) as total_trips,
            round(sum(fare_amount), 2) as total_revenue,
            round(avg(fare_amount), 2) as avg_fare_per_trip,
            round(avg(trip_distance), 2) as avg_distance
        FROM read_parquet('s3://silver/cleaned_taxi_data.parquet')
        GROUP BY vendor_id
    ) TO 's3://gold/vendor_performance.parquet' (FORMAT 'PARQUET');
"""

print("Creating Gold Layer metrics...")
conn.execute(gold_sql)

# Log a preview for the architect
stats = conn.execute(
    "SELECT * FROM 's3://gold/vendor_performance.parquet'"
).df()
print(stats)

Creating Gold Layer metrics...
   vendor_id  total_trips  total_revenue  avg_fare_per_trip  avg_distance
0          1      2554329   4.631680e+07              18.13          4.17
1          2      9056074   1.686934e+08              18.63          3.26
2          7        60655   9.657965e+05              15.92          2.77


In [4]:
import duckdb

# Connect to a local DuckDB file so your 'metadata' is saved
con = duckdb.connect('analysis.duckdb')

# Setup the connection to your local MinIO
con.execute("""
    INSTALL s3;
    LOAD s3;
    SET s3_endpoint='localhost:9000'; -- 'localhost' if running outside Docker
    SET s3_access_key_id='admin';
    SET s3_secret_access_key='password123';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

# 3. Pull the Silver data into a Pandas DataFrame for easy viewing
df = con.execute("SELECT * FROM 's3://silver/cleaned_taxi_data.parquet' LIMIT 10").df()

# Display the data
print("Preview of Cleaned Silver Data:")
display(df)

Preview of Cleaned Silver Data:


,vendor_id,pickup_time,dropoff_time,passenger_count,trip_distance,fare_amount
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.60,10.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.50,5.1
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.60,5.1
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,7.2
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,5.8
5,2,2025-01-01 00:48:24,2025-01-01 01:08:26,2,2.63,19.1
6,2,2025-01-01 00:00:02,2025-01-01 00:09:36,1,1.71,11.4
7,2,2025-01-01 00:20:28,2025-01-01 00:28:04,1,2.29,11.4
8,2,2025-01-01 00:33:58,2025-01-01 00:37:23,1,0.56,5.8
9,2,2025-01-01 00:42:40,2025-01-01 00:55:38,3,1.99,14.2


In [6]:
import duckdb

# Connect to a local DuckDB file so your 'metadata' is saved
con = duckdb.connect('analysis.duckdb')

# Setup the connection to your local MinIO
con.execute("""
    INSTALL s3;
    LOAD s3;
    SET s3_endpoint='localhost:9000'; -- 'localhost' if running outside Docker
    SET s3_access_key_id='admin';
    SET s3_secret_access_key='password123';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

# 3. Pull the Silver data into a Pandas DataFrame for easy viewing
df = con.execute("SELECT * FROM 's3://gold/vendor_performance.parquet' LIMIT 10").df()

# Display the data
print("Preview of Cleaned Silver Data:")
display(df)

Preview of Cleaned Silver Data:


,vendor_id,total_trips,total_revenue,avg_fare_per_trip,avg_distance
0,1,2554329,4.631680e+07,18.13,4.17
1,2,9056074,1.686934e+08,18.63,3.26
2,7,60655,9.657965e+05,15.92,2.77
